# Split a CSV by season

## Goal
Split any CSV containing a datetime column into winter, spring, summer, and fall files using the same month mapping as `data_processing/process_data.py`. All original columns and row values are preserved.

## Setup

Edit only the parameters in the next cell when using a different dataset. Set `WRITE_OUTPUTS = True` when you are ready to save the seasonal CSVs.

In [29]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()

# ----- Change these values as needed -----
INPUT_CSV = ROOT / 'data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All.csv'
DATETIME_COLUMN = 'dt'
OUTPUT_DIR = INPUT_CSV.parent / f'{INPUT_CSV.stem}_by_season'
WRITE_OUTPUTS = True
# -----------------------------------------

SEASON_MONTHS = {
    'winter': (12, 1, 2),
    'spring': (3, 4, 5),
    'summer': (6, 7, 8),
    'fall': (9, 10, 11),
}

## Steps

### 1. Load and validate the input

In [30]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

data = pd.read_csv(INPUT_CSV)
if DATETIME_COLUMN not in data.columns:
    raise KeyError(
        f"Datetime column '{DATETIME_COLUMN}' was not found. "
        f'Available columns: {data.columns.tolist()}'
    )

parsed_datetimes = pd.to_datetime(data[DATETIME_COLUMN], errors='coerce', utc=True)
invalid_datetime_mask = parsed_datetimes.isna()
if invalid_datetime_mask.any():
    examples = data.loc[invalid_datetime_mask, DATETIME_COLUMN].head(5).tolist()
    raise ValueError(
        f'{invalid_datetime_mask.sum()} rows have invalid datetimes in {DATETIME_COLUMN!r}. '
        f'Examples: {examples}'
    )

print(f'Loaded {len(data):,} rows from {INPUT_CSV}')
print(f'Date range: {parsed_datetimes.min()} through {parsed_datetimes.max()}')

Loaded 92 rows from /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All.csv
Date range: 2022-05-23 00:00:00+00:00 through 2025-09-22 00:00:00+00:00


### 2. Split rows by season

In [31]:
def split_by_season(frame, datetimes):
    """Return copies of frame grouped by the process_data.py season mapping."""
    return {
        season: frame.loc[datetimes.dt.month.isin(months)].copy()
        for season, months in SEASON_MONTHS.items()
    }

seasonal_data = split_by_season(data, parsed_datetimes)
season_summary = pd.DataFrame(
    {
        'season': list(seasonal_data),
        'months': [', '.join(map(str, SEASON_MONTHS[name])) for name in seasonal_data],
        'rows': [len(frame) for frame in seasonal_data.values()],
    }
)
season_summary['percent_of_input'] = 100 * season_summary['rows'] / len(data)
season_summary

,season,months,rows,percent_of_input
0,winter,"12, 1, 2",0,0.000000
1,spring,"3, 4, 5",9,9.782609
2,summer,"6, 7, 8",73,79.347826
3,fall,"9, 10, 11",10,10.869565


### 3. Save the seasonal files

In [32]:
output_paths = {
    season: OUTPUT_DIR / f'{INPUT_CSV.stem}_{season}.csv'
    for season in seasonal_data
}

if WRITE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for season, frame in seasonal_data.items():
        frame.to_csv(output_paths[season], index=False)
        print(f'Wrote {len(frame):,} rows to {output_paths[season]}')
else:
    print('Preview only: set WRITE_OUTPUTS = True to save files.')
    for path in output_paths.values():
        print(path)

Wrote 0 rows to /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All_by_season/Canister_Filtered_No_Smoke_All_winter.csv
Wrote 9 rows to /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All_by_season/Canister_Filtered_No_Smoke_All_spring.csv
Wrote 73 rows to /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All_by_season/Canister_Filtered_No_Smoke_All_summer.csv
Wrote 10 rows to /Users/isaacsudweeks/Desktop/Work/Improved_Isopleths/data/utah/data_up_to_2025/new_canister_breakout_2026_06_23/Filtered_Canister_data/PM2.5_Smoke/No_Smoke/Canister_Filtered_No_Smoke_All_by_season/Canister_Filtered_No

## Checks

Confirm that every input row appears in exactly one seasonal group and that the original schema and row order within each season are preserved.

In [33]:
assert set(SEASON_MONTHS) == {'winter', 'spring', 'summer', 'fall'}
assert sorted(month for months in SEASON_MONTHS.values() for month in months) == list(range(1, 13))
assert sum(len(frame) for frame in seasonal_data.values()) == len(data)
assert all(list(frame.columns) == list(data.columns) for frame in seasonal_data.values())

input_indices = set(data.index)
season_indices = [set(frame.index) for frame in seasonal_data.values()]
assert set().union(*season_indices) == input_indices
assert sum(len(indices) for indices in season_indices) == len(input_indices)

if WRITE_OUTPUTS:
    for season, path in output_paths.items():
        written = pd.read_csv(path)
        assert list(written.columns) == list(data.columns)
        assert len(written) == len(seasonal_data[season])

print(f'Checks passed: all {len(data):,} rows belong to exactly one season.')

Checks passed: all 92 rows belong to exactly one season.


## Next Steps

To split another dataset, update `INPUT_CSV`, `DATETIME_COLUMN`, and optionally `OUTPUT_DIR`, set `WRITE_OUTPUTS = True`, then run all cells. Winter follows the existing project convention and combines December, January, and February without assigning a separate winter-year label.